# 04. Comparación de embeddings con similitud coseno y métricas complementarias

Este notebook corresponde al **paso 04** del pipeline.

Toma como base lo generado en:

- **Paso 02:** detección del perro y clasificación de raza.
- **Paso 03:** extracción de características visuales y embeddings.

La métrica principal de comparación será la **similitud coseno**, siguiendo la recomendación de la profesora.  
Además, se calculan métricas complementarias para validar la consistencia de los resultados:

- Distancia euclidiana.
- Distancia Manhattan.
- Same-breed Top-K Accuracy.
- Similitud intra-raza.
- Similitud inter-raza.
- Detección de posibles outliers.

---

## Idea principal

Cada imagen fue transformada en un embedding visual en el paso 03.  
Ahora comparamos esos embeddings para saber qué imágenes son más parecidas visualmente.

La pregunta principal del paso 04 es:

> ¿Los embeddings permiten encontrar imágenes visualmente similares, especialmente dentro de la misma raza?

## Métrica principal: similitud coseno

La similitud coseno mide la cercanía entre dos vectores considerando el ángulo entre ellos.

\[
\cos(\theta)=\frac{A\cdot B}{\|A\|\|B\|}
\]

Donde:

| Elemento | Significado |
|---|---|
| `A` | Embedding de la imagen 1 |
| `B` | Embedding de la imagen 2 |
| `A · B` | Producto punto entre ambos vectores |
| `||A||`, `||B||` | Norma o magnitud de cada vector |
| `cos(θ)` | Similitud entre ambos embeddings |

Interpretación:

| Valor de coseno | Interpretación |
|---:|---|
| Cercano a 1.0 | Muy similares |
| Cercano a 0.0 | Poco similares |
| Negativo | Muy diferentes |

En este proyecto, la similitud coseno será la métrica principal para encontrar perros visualmente parecidos.

## Métricas complementarias

Aunque la métrica principal es coseno, también se calculan métricas complementarias:

| Métrica | Interpretación | Uso |
|---|---|---|
| Similitud coseno | Más alto = más parecido | Principal |
| Distancia euclidiana | Más bajo = más parecido | Complementaria |
| Distancia Manhattan | Más bajo = más parecido | Complementaria |
| Same-breed Top-K | Vecinos del mismo tipo de raza | Indicador de calidad |
| Outlier score | Baja similitud con su propia raza | Revisión de casos atípicos |

La interpretación principal se basa en coseno. Las demás métricas ayudan a validar si los resultados son consistentes.

In [ ]:
# 0. Instalación de dependencias

!pip install -q numpy pandas scikit-learn matplotlib pillow tqdm

In [ ]:
# 1. Imports y configuración general

from pathlib import Path
import os
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# 2. Montar Google Drive

from google.colab import drive

if Path("/content/drive/MyDrive").exists():
    print("Google Drive ya está montado.")
else:
    drive.mount("/content/drive")

In [ ]:
# 3. Rutas del proyecto

PROJECT_PATH = Path("/content/drive/MyDrive/proyecto_integrador")

DATA_PROCESSED_PATH = PROJECT_PATH / "data_processed"

# Paso 02
STEP02_PATH = DATA_PROCESSED_PATH / "step02_detection_classification"
STEP02_REPORTS_PATH = STEP02_PATH / "reports"

# Paso 03
STEP03_PATH = DATA_PROCESSED_PATH / "step03_visual_features_embeddings"
STEP03_REPORTS_PATH = STEP03_PATH / "reports"
STEP03_ARRAYS_PATH = STEP03_PATH / "arrays"

# Paso 04
STEP04_PATH = DATA_PROCESSED_PATH / "step04_embedding_similarity"
STEP04_REPORTS_PATH = STEP04_PATH / "reports"
STEP04_PLOTS_PATH = STEP04_PATH / "plots"
STEP04_EXAMPLES_PATH = STEP04_PATH / "example_grids"

for p in [STEP04_PATH, STEP04_REPORTS_PATH, STEP04_PLOTS_PATH, STEP04_EXAMPLES_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_PATH:", PROJECT_PATH)
print("STEP03_ARRAYS_PATH:", STEP03_ARRAYS_PATH)
print("STEP04_PATH:", STEP04_PATH)

In [ ]:
# 4. Configuración del experimento

INPUT_VARIANT = "yolo_crop"
# INPUT_VARIANT = "full_curated_image"

TOP_K = 5
TOP_K_EXTENDED = 10

# Para evitar cálculos enormes de todos contra todos en datasets grandes,
# se usa muestreo en algunos análisis.
MAX_PAIRS_PER_BREED = 5000
INTER_BREED_SAMPLE_PAIRS = 20000

EMBEDDINGS_PATH = STEP03_ARRAYS_PATH / f"step03_{INPUT_VARIANT}_image_embeddings_l2_normalized.npy"
EMBEDDINGS_METADATA_PATH = STEP03_REPORTS_PATH / f"step03_{INPUT_VARIANT}_embeddings_metadata.csv"
VISUAL_FEATURES_PATH = STEP03_REPORTS_PATH / f"step03_{INPUT_VARIANT}_visual_features.csv"

print("INPUT_VARIANT:", INPUT_VARIANT)
print("EMBEDDINGS_PATH:", EMBEDDINGS_PATH)
print("EMBEDDINGS_METADATA_PATH:", EMBEDDINGS_METADATA_PATH)
print("VISUAL_FEATURES_PATH:", VISUAL_FEATURES_PATH)

# Carga y validación de entradas

En esta sección se cargan los embeddings normalizados generados en el paso 03 y su metadata.

Los embeddings normalizados permiten usar similitud coseno de forma directa.  
Aun así, se calcula explícitamente `cosine_similarity ` para que la métrica sea clara.

In [ ]:
# 5. Cargar embeddings y metadata del paso 03

if not EMBEDDINGS_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo de embeddings: {EMBEDDINGS_PATH}\n"
        "Primero ejecuta el paso 03."
    )

if not EMBEDDINGS_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró la metadata de embeddings: {EMBEDDINGS_METADATA_PATH}\n"
        "Primero ejecuta el paso 03."
    )

embeddings = np.load(EMBEDDINGS_PATH)
metadata_df = pd.read_csv(EMBEDDINGS_METADATA_PATH)

print("Embeddings shape:", embeddings.shape)
print("Metadata rows:", len(metadata_df))
print("Columnas metadata:", metadata_df.columns.tolist())

display(metadata_df.head())

In [ ]:
# 6. Limpieza y alineación metadata-embeddings

# Nos quedamos solo con embeddings válidos.
if "embedding_status" in metadata_df.columns:
    metadata_ok_df = metadata_df[metadata_df["embedding_status"] == "ok"].copy()
else:
    metadata_ok_df = metadata_df.copy()

if len(metadata_ok_df) != embeddings.shape[0]:
    print("Advertencia: metadata_ok_df y embeddings no tienen la misma longitud.")
    print("metadata_ok_df:", len(metadata_ok_df))
    print("embeddings:", embeddings.shape[0])
    print("Se alineará usando las primeras filas válidas.")
    metadata_ok_df = metadata_ok_df.head(embeddings.shape[0]).copy()

metadata_ok_df = metadata_ok_df.reset_index(drop=True)

required_cols = ["image_input_path", "breed"]
missing_cols = [c for c in required_cols if c not in metadata_ok_df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas necesarias en metadata: {missing_cols}")

# Asegurar normalización L2.
embeddings_l2 = normalize(embeddings, norm="l2")

print("Metadata final:", metadata_ok_df.shape)
print("Embeddings final:", embeddings_l2.shape)
print("Razas:", metadata_ok_df["breed"].nunique())

# Vecinos más similares con similitud coseno

Para cada imagen se buscan sus vecinos más cercanos según similitud coseno.

Este análisis responde:

> Dada una imagen, ¿cuáles son las imágenes más parecidas visualmente?

In [ ]:
# Ajuste de rutas de imágenes para visualización usando content y no drive, para hacer mas eficiente el procesamiento

from pathlib import Path
import subprocess

DRIVE_CROPS_PATH = Path("/content/drive/MyDrive/proyecto_integrador/data_processed/step02_detection_classification/dog_crops_from_curated_v2")
LOCAL_CROPS_PATH = Path("/content/dog_crops_from_curated_v2")

DRIVE_CURATED_PATH = Path("/content/drive/MyDrive/proyecto_integrador/images_curated_rgb_v2")
LOCAL_CURATED_PATH = Path("/content/images_curated_rgb_v2")

def ensure_local_folder(drive_path, local_path):
    if not local_path.exists():
        print(f"Copiando {drive_path} → {local_path}")
        subprocess.run(["cp", "-r", str(drive_path), str(local_path)], check=True)
    else:
        print(f"Ya existe: {local_path}")

if INPUT_VARIANT == "yolo_crop":
    ensure_local_folder(DRIVE_CROPS_PATH, LOCAL_CROPS_PATH)

    metadata_ok_df["image_input_path"] = metadata_ok_df["image_input_path"].astype(str).str.replace(
        str(DRIVE_CROPS_PATH),
        str(LOCAL_CROPS_PATH),
        regex=False
    )

    metadata_ok_df["image_input_path"] = metadata_ok_df["image_input_path"].astype(str).str.replace(
        "/content/dog_crops_from_curated_v2",
        str(LOCAL_CROPS_PATH),
        regex=False
    )

elif INPUT_VARIANT == "full_curated_image":
    ensure_local_folder(DRIVE_CURATED_PATH, LOCAL_CURATED_PATH)

    metadata_ok_df["image_input_path"] = metadata_ok_df["image_input_path"].astype(str).str.replace(
        str(DRIVE_CURATED_PATH),
        str(LOCAL_CURATED_PATH),
        regex=False
    )

    metadata_ok_df["image_input_path"] = metadata_ok_df["image_input_path"].astype(str).str.replace(
        "/content/images_curated_rgb_v2",
        str(LOCAL_CURATED_PATH),
        regex=False
    )

metadata_ok_df["image_exists"] = metadata_ok_df["image_input_path"].apply(lambda p: Path(p).exists())

print(metadata_ok_df["image_exists"].value_counts())

display(metadata_ok_df[["image_input_path", "breed", "image_exists"]].head())

In [ ]:
# 7. Función para obtener vecinos Top-K por similitud coseno

def topk_cosine_neighbors(query_index, embeddings_normalized, metadata, top_k=5):
    query_vector = embeddings_normalized[query_index].reshape(1, -1)
    similarities = cosine_similarity(query_vector, embeddings_normalized)[0]

    similarities[query_index] = -np.inf

    top_indices = np.argsort(similarities)[::-1][:top_k]

    query_breed = metadata.loc[query_index, "breed"]
    query_path = metadata.loc[query_index, "image_input_path"]

    rows = []
    for rank, idx in enumerate(top_indices, start=1):
        neighbor_breed = metadata.loc[idx, "breed"]
        rows.append({
            "query_index": query_index,
            "query_image": query_path,
            "query_breed": query_breed,
            "neighbor_rank": rank,
            "neighbor_index": int(idx),
            "neighbor_image": metadata.loc[idx, "image_input_path"],
            "neighbor_breed": neighbor_breed,
            "cosine_similarity": float(similarities[idx]),
            "same_breed": bool(query_breed == neighbor_breed)
        })

    return pd.DataFrame(rows)


# Ejemplo individual
example_neighbors_df = topk_cosine_neighbors(
    query_index=0,
    embeddings_normalized=embeddings_l2,
    metadata=metadata_ok_df,
    top_k=TOP_K_EXTENDED
)

display(example_neighbors_df)

In [ ]:
# 8. Calcular Top-K vecinos para todas las imágenes

def compute_topk_all(embeddings_normalized, metadata, top_k=5, batch_size=512):
    all_records = []
    n = len(metadata)

    for start in tqdm(range(0, n, batch_size)):
        end = min(start + batch_size, n)
        batch = embeddings_normalized[start:end]

        sim_batch = cosine_similarity(batch, embeddings_normalized)

        for local_i, global_i in enumerate(range(start, end)):
            sim_row = sim_batch[local_i]
            sim_row[global_i] = -np.inf

            top_indices = np.argsort(sim_row)[::-1][:top_k]
            query_breed = metadata.loc[global_i, "breed"]
            query_path = metadata.loc[global_i, "image_input_path"]

            for rank, neighbor_idx in enumerate(top_indices, start=1):
                neighbor_breed = metadata.loc[neighbor_idx, "breed"]

                all_records.append({
                    "query_index": int(global_i),
                    "query_image": query_path,
                    "query_breed": query_breed,
                    "neighbor_rank": rank,
                    "neighbor_index": int(neighbor_idx),
                    "neighbor_image": metadata.loc[neighbor_idx, "image_input_path"],
                    "neighbor_breed": neighbor_breed,
                    "cosine_similarity": float(sim_row[neighbor_idx]),
                    "same_breed": bool(query_breed == neighbor_breed)
                })

    return pd.DataFrame(all_records)


topk_neighbors_df = compute_topk_all(
    embeddings_normalized=embeddings_l2,
    metadata=metadata_ok_df,
    top_k=TOP_K_EXTENDED,
    batch_size=512
)

TOPK_NEIGHBORS_PATH = STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_cosine_topk_neighbors.csv"
topk_neighbors_df.to_csv(TOPK_NEIGHBORS_PATH, index=False)

print("Top-K vecinos guardado en:", TOPK_NEIGHBORS_PATH)
display(topk_neighbors_df.head(20))

In [ ]:
# 9. Indicadores Same-Breed Top-K

def same_breed_at_k(topk_df, k):
    subset = topk_df[topk_df["neighbor_rank"] <= k].copy()

    # Por cada imagen que consulta, revisa si al menos un vecino en Top-K es de la misma raza.
    per_query = (
        subset.groupby("query_index")["same_breed"]
        .max()
        .reset_index(name=f"same_breed_at_{k}")
    )

    return per_query[f"same_breed_at_{k}"].mean() * 100


def mean_same_breed_ratio_at_k(topk_df, k):
    subset = topk_df[topk_df["neighbor_rank"] <= k].copy()

    per_query = (
        subset.groupby("query_index")["same_breed"]
        .mean()
        .reset_index(name=f"same_breed_ratio_at_{k}")
    )

    return per_query[f"same_breed_ratio_at_{k}"].mean() * 100


same_breed_at_1 = same_breed_at_k(topk_neighbors_df, 1)
same_breed_at_5 = same_breed_at_k(topk_neighbors_df, 5)
same_breed_at_10 = same_breed_at_k(topk_neighbors_df, 10)

same_breed_ratio_at_5 = mean_same_breed_ratio_at_k(topk_neighbors_df, 5)
same_breed_ratio_at_10 = mean_same_breed_ratio_at_k(topk_neighbors_df, 10)

same_breed_metrics_df = pd.DataFrame([
    {"indicator": "same_breed_at_1_percent", "value": same_breed_at_1},
    {"indicator": "same_breed_at_5_percent", "value": same_breed_at_5},
    {"indicator": "same_breed_at_10_percent", "value": same_breed_at_10},
    {"indicator": "avg_same_breed_ratio_at_5_percent", "value": same_breed_ratio_at_5},
    {"indicator": "avg_same_breed_ratio_at_10_percent", "value": same_breed_ratio_at_10},
])

display(same_breed_metrics_df)

# Similitud intra-raza

En esta sección se comparan imágenes **dentro de la misma raza** usando similitud coseno.

Este análisis responde:

> ¿Qué tan parecidas son entre sí las imágenes de una misma raza?

Una raza con alta similitud promedio indica que sus imágenes son visualmente consistentes.  
Una raza con baja similitud promedio puede tener mucha variabilidad visual, poses difíciles, mala calidad o casos atípicos.

In [ ]:
# 10. Resumen de similitud coseno intra-raza

breed_summary_records = []
same_breed_pair_records = []

rng = np.random.default_rng(SEED)

for breed, group in tqdm(metadata_ok_df.groupby("breed")):
    indices = group.index.to_numpy()

    if len(indices) < 2:
        continue

    breed_embeddings = embeddings_l2[indices]
    sim_matrix = cosine_similarity(breed_embeddings)

    # Valores del triángulo superior, excluyendo diagonal.
    upper_i, upper_j = np.triu_indices_from(sim_matrix, k=1)
    sim_values = sim_matrix[upper_i, upper_j]

    # Guardar muestra de pares para no generar archivos demasiado grandes.
    total_pairs = len(sim_values)

    if total_pairs > MAX_PAIRS_PER_BREED:
        sampled_positions = rng.choice(total_pairs, size=MAX_PAIRS_PER_BREED, replace=False)
    else:
        sampled_positions = np.arange(total_pairs)

    for pos in sampled_positions:
        local_i = upper_i[pos]
        local_j = upper_j[pos]
        global_i = indices[local_i]
        global_j = indices[local_j]

        same_breed_pair_records.append({
            "breed": breed,
            "image_a_index": int(global_i),
            "image_b_index": int(global_j),
            "image_a": metadata_ok_df.loc[global_i, "image_input_path"],
            "image_b": metadata_ok_df.loc[global_j, "image_input_path"],
            "cosine_similarity": float(sim_matrix[local_i, local_j])
        })

    breed_summary_records.append({
        "breed": breed,
        "num_images": int(len(indices)),
        "num_pairs_total": int(total_pairs),
        "num_pairs_saved_sample": int(len(sampled_positions)),
        "avg_cosine_similarity_intra_breed": float(np.mean(sim_values)),
        "median_cosine_similarity_intra_breed": float(np.median(sim_values)),
        "min_cosine_similarity_intra_breed": float(np.min(sim_values)),
        "max_cosine_similarity_intra_breed": float(np.max(sim_values)),
        "std_cosine_similarity_intra_breed": float(np.std(sim_values))
    })

same_breed_pairs_df = pd.DataFrame(same_breed_pair_records)
breed_similarity_summary_df = pd.DataFrame(breed_summary_records).sort_values(
    "avg_cosine_similarity_intra_breed",
    ascending=False
)

SAME_BREED_PAIRS_PATH = STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_same_breed_cosine_pairs_sample.csv"
BREED_SUMMARY_PATH = STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_same_breed_cosine_summary.csv"

same_breed_pairs_df.to_csv(SAME_BREED_PAIRS_PATH, index=False)
breed_similarity_summary_df.to_csv(BREED_SUMMARY_PATH, index=False)

print("Pares intra-raza guardados:", SAME_BREED_PAIRS_PATH)
print("Resumen por raza guardado:", BREED_SUMMARY_PATH)

display(breed_similarity_summary_df.head(10))
display(breed_similarity_summary_df.tail(10))

In [ ]:
# 11. Visualizar razas con mayor y menor similitud intra-raza

plt.figure(figsize=(12, 5))
breed_similarity_summary_df.head(20).set_index("breed")["avg_cosine_similarity_intra_breed"].plot(kind="bar")
plt.title("Top 20 razas con mayor similitud promedio intra-raza")
plt.ylabel("Similitud coseno promedio")
plt.xlabel("Raza")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(STEP04_PLOTS_PATH / f"step04_{INPUT_VARIANT}_top20_highest_intra_breed_similarity.png", dpi=150)
plt.show()

plt.figure(figsize=(12, 5))
breed_similarity_summary_df.tail(20).set_index("breed")["avg_cosine_similarity_intra_breed"].plot(kind="bar")
plt.title("Top 20 razas con menor similitud promedio intra-raza")
plt.ylabel("Similitud coseno promedio")
plt.xlabel("Raza")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(STEP04_PLOTS_PATH / f"step04_{INPUT_VARIANT}_top20_lowest_intra_breed_similarity.png", dpi=150)
plt.show()

# Comparación intra-raza vs inter-raza

Además de comparar imágenes de la misma raza, conviene comparar imágenes de razas distintas.

La hipótesis esperada es:

> La similitud promedio dentro de la misma raza debería ser mayor que la similitud promedio entre razas diferentes.

Si esto ocurre, significa que los embeddings capturan información útil para separar visualmente las razas.

In [ ]:
# 12. Muestreo de similitud inter-raza

n = len(metadata_ok_df)
inter_records = []

# Para eficiencia, se muestrean pares aleatorios de razas diferentes.
attempts = 0
max_attempts = INTER_BREED_SAMPLE_PAIRS * 10

while len(inter_records) < INTER_BREED_SAMPLE_PAIRS and attempts < max_attempts:
    i, j = rng.integers(0, n, size=2)
    attempts += 1

    if i == j:
        continue

    breed_i = metadata_ok_df.loc[i, "breed"]
    breed_j = metadata_ok_df.loc[j, "breed"]

    if breed_i == breed_j:
        continue

    sim = float(cosine_similarity(
        embeddings_l2[i].reshape(1, -1),
        embeddings_l2[j].reshape(1, -1)
    )[0, 0])

    inter_records.append({
        "image_a_index": int(i),
        "image_b_index": int(j),
        "breed_a": breed_i,
        "breed_b": breed_j,
        "image_a": metadata_ok_df.loc[i, "image_input_path"],
        "image_b": metadata_ok_df.loc[j, "image_input_path"],
        "cosine_similarity": sim
    })

inter_breed_pairs_df = pd.DataFrame(inter_records)

INTER_BREED_PAIRS_PATH = STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_inter_breed_cosine_pairs_sample.csv"
inter_breed_pairs_df.to_csv(INTER_BREED_PAIRS_PATH, index=False)

print("Pares inter-raza guardados:", INTER_BREED_PAIRS_PATH)
display(inter_breed_pairs_df.head())

In [ ]:
# 13. Comparación estadística intra vs inter

intra_values = same_breed_pairs_df["cosine_similarity"].dropna().values
inter_values = inter_breed_pairs_df["cosine_similarity"].dropna().values

intra_inter_summary_df = pd.DataFrame([
    {
        "group": "intra_breed",
        "num_pairs": len(intra_values),
        "avg_cosine_similarity": float(np.mean(intra_values)),
        "median_cosine_similarity": float(np.median(intra_values)),
        "std_cosine_similarity": float(np.std(intra_values)),
        "min_cosine_similarity": float(np.min(intra_values)),
        "max_cosine_similarity": float(np.max(intra_values))
    },
    {
        "group": "inter_breed",
        "num_pairs": len(inter_values),
        "avg_cosine_similarity": float(np.mean(inter_values)),
        "median_cosine_similarity": float(np.median(inter_values)),
        "std_cosine_similarity": float(np.std(inter_values)),
        "min_cosine_similarity": float(np.min(inter_values)),
        "max_cosine_similarity": float(np.max(inter_values))
    }
])

separation_margin = float(np.mean(intra_values) - np.mean(inter_values))

INTRA_INTER_SUMMARY_PATH = STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_intra_vs_inter_cosine_summary.csv"
intra_inter_summary_df.to_csv(INTRA_INTER_SUMMARY_PATH, index=False)

print("Margen de separación promedio intra - inter:", separation_margin)
display(intra_inter_summary_df)

plt.figure(figsize=(8, 5))
plt.hist(intra_values, bins=40, alpha=0.6, label="Intra-raza")
plt.hist(inter_values, bins=40, alpha=0.6, label="Inter-raza")
plt.title("Distribución de similitud coseno: intra-raza vs inter-raza")
plt.xlabel("Similitud coseno")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.savefig(STEP04_PLOTS_PATH / f"step04_{INPUT_VARIANT}_intra_vs_inter_cosine_hist.png", dpi=150)
plt.show()

# Métricas complementarias: euclidiana y Manhattan

La métrica principal sigue siendo coseno.  
Sin embargo, se calculan distancias euclidiana y Manhattan como validación complementaria.

Interpretación:

| Métrica | Mejor valor |
|---|---|
| Similitud coseno | Más alto |
| Distancia euclidiana | Más bajo |
| Distancia Manhattan | Más bajo |

In [ ]:
# 14. Métricas complementarias sobre vecinos Top-K
# Para evitar calcular matrices enormes completas, se calculan distancias solo para los pares Top-K encontrados por coseno.

complementary_records = []

for _, row in tqdm(topk_neighbors_df.iterrows(), total=len(topk_neighbors_df)):
    i = int(row["query_index"])
    j = int(row["neighbor_index"])

    vec_i = embeddings_l2[i].reshape(1, -1)
    vec_j = embeddings_l2[j].reshape(1, -1)

    euclidean_distance = float(pairwise_distances(vec_i, vec_j, metric="euclidean")[0, 0])
    manhattan_distance = float(pairwise_distances(vec_i, vec_j, metric="manhattan")[0, 0])

    record = row.to_dict()
    record["euclidean_distance"] = euclidean_distance
    record["manhattan_distance"] = manhattan_distance

    complementary_records.append(record)

topk_with_distances_df = pd.DataFrame(complementary_records)

TOPK_WITH_DISTANCES_PATH = STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_topk_cosine_with_complementary_distances.csv"
topk_with_distances_df.to_csv(TOPK_WITH_DISTANCES_PATH, index=False)

print("Top-K con métricas complementarias guardado en:", TOPK_WITH_DISTANCES_PATH)
display(topk_with_distances_df.head(20))

In [ ]:
# 15. Resumen de métricas complementarias

complementary_summary_df = (
    topk_with_distances_df
    .groupby("same_breed")
    .agg(
        count=("same_breed", "size"),
        avg_cosine_similarity=("cosine_similarity", "mean"),
        avg_euclidean_distance=("euclidean_distance", "mean"),
        avg_manhattan_distance=("manhattan_distance", "mean")
    )
    .reset_index()
)

COMPLEMENTARY_SUMMARY_PATH = STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_complementary_metrics_summary.csv"
complementary_summary_df.to_csv(COMPLEMENTARY_SUMMARY_PATH, index=False)

display(complementary_summary_df)
print("Resumen complementario guardado:", COMPLEMENTARY_SUMMARY_PATH)

# Detección de outliers por raza

Un outlier puede ser una imagen que pertenece a una raza, pero cuyo embedding no se parece mucho al resto de imágenes de esa misma raza.

La regla usada será:

> Si una imagen tiene baja similitud promedio con sus vecinos de la misma raza, se marca como posible outlier.

Esto puede indicar:

- Imagen difícil.
- Mala calidad visual.
- Perro parcialmente visible.
- Fondo dominante.
- Etiqueta incorrecta.
- Raza con alta variabilidad visual.

In [ ]:
# 16. Calcular outlier score por raza

outlier_records = []

for breed, group in tqdm(metadata_ok_df.groupby("breed")):
    indices = group.index.to_numpy()

    if len(indices) < 3:
        continue

    breed_embeddings = embeddings_l2[indices]
    sim_matrix = cosine_similarity(breed_embeddings)

    # Excluir diagonal
    np.fill_diagonal(sim_matrix, np.nan)

    # Promedio de similitud de cada imagen con las demás de su raza
    avg_similarity_to_same_breed = np.nanmean(sim_matrix, axis=1)

    for local_idx, global_idx in enumerate(indices):
        outlier_records.append({
            "image_index": int(global_idx),
            "image_input_path": metadata_ok_df.loc[global_idx, "image_input_path"],
            "breed": breed,
            "avg_similarity_to_same_breed": float(avg_similarity_to_same_breed[local_idx]),
            "outlier_score": float(1 - avg_similarity_to_same_breed[local_idx])
        })

outliers_df = pd.DataFrame(outlier_records)

# Definir posibles outliers: percentil 95 del outlier_score dentro de cada raza
outliers_df["breed_outlier_threshold"] = (
    outliers_df.groupby("breed")["outlier_score"]
    .transform(lambda x: x.quantile(0.95))
)

outliers_df["is_outlier_candidate"] = (
    outliers_df["outlier_score"] >= outliers_df["breed_outlier_threshold"]
)

outlier_candidates_df = outliers_df[outliers_df["is_outlier_candidate"]].sort_values(
    "outlier_score",
    ascending=False
)

OUTLIERS_PATH = STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_outlier_candidates.csv"
outliers_df.to_csv(STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_outlier_scores_all.csv", index=False)
outlier_candidates_df.to_csv(OUTLIERS_PATH, index=False)

print("Outliers candidatos guardados:", OUTLIERS_PATH)
print("Total candidatos:", len(outlier_candidates_df))

display(outlier_candidates_df.head(20))

In [ ]:
# 17. Visualizar ejemplos de outliers

def show_image_grid(paths, titles=None, cols=5, figsize=(15, 8), save_path=None):
    paths = list(paths)
    n = len(paths)

    if n == 0:
        print("No hay imágenes para mostrar.")
        return

    rows = math.ceil(n / cols)
    plt.figure(figsize=figsize)

    for i, p in enumerate(paths):
        try:
            img = Image.open(p).convert("RGB")
            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            if titles is not None:
                plt.title(titles[i], fontsize=8)
            plt.axis("off")
        except Exception as e:
            print("Error mostrando imagen:", p, e)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)

    plt.show()


example_outliers = outlier_candidates_df.head(10)
outlier_paths = example_outliers["image_input_path"].tolist()
outlier_titles = [
    f"{row['breed']}\nscore={row['outlier_score']:.2f}"
    for _, row in example_outliers.iterrows()
]

show_image_grid(
    outlier_paths,
    titles=outlier_titles,
    cols=5,
    figsize=(15, 6),
    save_path=STEP04_PLOTS_PATH / f"step04_{INPUT_VARIANT}_outlier_examples.png"
)

# Visualización de vecinos similares

Esta sección muestra una imagen consulta y sus vecinos visuales más similares usando coseno.

Sirve para explicar visualmente cómo funciona la comparación de embeddings.

In [ ]:
# 18. Visualizar vecinos similares de una imagen

def visualize_query_neighbors(query_index, topk_df, metadata, top_k=5, save=True):
    query_row = metadata.loc[query_index]
    query_path = query_row["image_input_path"]
    query_breed = query_row["breed"]

    neighbors = topk_df[
        (topk_df["query_index"] == query_index) &
        (topk_df["neighbor_rank"] <= top_k)
    ].sort_values("neighbor_rank")

    paths = [query_path] + neighbors["neighbor_image"].tolist()
    titles = [f"Query\n{query_breed}"]

    for _, row in neighbors.iterrows():
        titles.append(
            f"Rank {int(row['neighbor_rank'])}\n"
            f"{row['neighbor_breed']}\n"
            f"cos={row['cosine_similarity']:.2f}"
        )

    save_path = None
    if save:
        save_path = STEP04_EXAMPLES_PATH / f"query_{query_index}_top{top_k}_neighbors.png"

    show_image_grid(
        paths,
        titles=titles,
        cols=top_k + 1,
        figsize=(3 * (top_k + 1), 3.5),
        save_path=save_path
    )

# Ejemplo
visualize_query_neighbors(
    query_index=0,
    topk_df=topk_neighbors_df,
    metadata=metadata_ok_df,
    top_k=5,
    save=True
)

In [ ]:
# 19. Guardar varios ejemplos de vecinos similares

# Seleccionamos algunas imágenes con vecinos de la misma raza y otras con vecinos distintos.
queries_with_same_breed = (
    topk_neighbors_df[topk_neighbors_df["neighbor_rank"] == 1]
    .query("same_breed == True")["query_index"]
    .head(3)
    .tolist()
)

queries_without_same_breed = (
    topk_neighbors_df[topk_neighbors_df["neighbor_rank"] == 1]
    .query("same_breed == False")["query_index"]
    .head(3)
    .tolist()
)

example_queries = queries_with_same_breed + queries_without_same_breed

for q_idx in example_queries:
    visualize_query_neighbors(
        query_index=int(q_idx),
        topk_df=topk_neighbors_df,
        metadata=metadata_ok_df,
        top_k=5,
        save=True
    )

print("Ejemplos guardados en:", STEP04_EXAMPLES_PATH)

# Indicadores finales del paso 04

Esta sección consolida los indicadores principales del análisis de similitud.

In [ ]:
# 20. Indicadores finales del paso 04

step04_indicators = {
    "input_variant": INPUT_VARIANT,
    "num_embeddings_evaluated": int(len(metadata_ok_df)),
    "embedding_dimension": int(embeddings_l2.shape[1]),
    "num_breeds": int(metadata_ok_df["breed"].nunique()),
    "top_k_main": TOP_K,
    "same_breed_at_1_percent": same_breed_at_1,
    "same_breed_at_5_percent": same_breed_at_5,
    "same_breed_at_10_percent": same_breed_at_10,
    "avg_same_breed_ratio_at_5_percent": same_breed_ratio_at_5,
    "avg_same_breed_ratio_at_10_percent": same_breed_ratio_at_10,
    "avg_intra_breed_cosine_similarity": float(np.mean(intra_values)),
    "avg_inter_breed_cosine_similarity": float(np.mean(inter_values)),
    "intra_minus_inter_cosine_margin": separation_margin,
    "num_same_breed_pairs_sampled": int(len(same_breed_pairs_df)),
    "num_inter_breed_pairs_sampled": int(len(inter_breed_pairs_df)),
    "num_outlier_candidates": int(len(outlier_candidates_df)),
}

step04_indicators_df = pd.DataFrame(
    [{"indicator": k, "value": v} for k, v in step04_indicators.items()]
)

STEP04_INDICATORS_PATH = STEP04_REPORTS_PATH / f"step04_{INPUT_VARIANT}_similarity_indicators.csv"
step04_indicators_df.to_csv(STEP04_INDICATORS_PATH, index=False)

display(step04_indicators_df)
print("Indicadores guardados:", STEP04_INDICATORS_PATH)

# Análisis de métricas finales Paso 04: Comparación de embeddings

En este paso se compararon los embeddings generados en el Paso 03 utilizando **similitud coseno** como métrica principal. El objetivo fue evaluar si las representaciones profundas generadas por EfficientNet permiten encontrar imágenes visualmente similares, especialmente dentro de la misma raza.

Además de la similitud coseno, se utilizaron métricas complementarias como distancia euclidiana y distancia Manhattan para validar la consistencia de los resultados.

Este análisis permite pasar de un sistema que solo clasifica razas a un sistema capaz de **buscar perros visualmente parecidos**.

## 1. Indicadores generales del Paso 04

| Indicador | Resultado |
|---|---:|
| Variante de entrada | yolo_crop |
| Embeddings evaluados | 18,807 |
| Dimensión del embedding | 256 |
| Razas evaluadas | 120 |
| Top-K principal | 5 |
| Pares intra-raza analizados | 595,823 |
| Pares inter-raza muestreados | 20,000 |
| Candidatos a outlier | 1,002 |

El análisis se realizó sobre **18,807 embeddings**, correspondientes a las imágenes donde YOLO detectó correctamente un perro y se generó un crop válido.

Cada imagen fue representada mediante un vector de **256 dimensiones**, extraído desde una capa intermedia del modelo EfficientNet entrenado en el Paso 02. Las **120 razas** permanecen representadas en esta etapa, lo cual indica que el proceso de extracción de embeddings no eliminó clases completas.

La variante utilizada fue `yolo_crop`, por lo que las comparaciones se realizaron sobre recortes del perro generados por YOLO, no sobre la imagen completa.

## 2. Análisis de vecinos similares con similitud coseno

| Indicador | Resultado |
|---|---:|
| Same-breed at 1 | 74.90% |
| Same-breed at 5 | 89.94% |
| Same-breed at 10 | 93.15% |
| Promedio de vecinos de la misma raza en Top-5 | 72.76% |
| Promedio de vecinos de la misma raza en Top-10 | 71.54% |

La métrica **Same-breed at K** mide si, dentro de los vecinos más similares de una imagen, aparece al menos una imagen de la misma raza.

El resultado de **Same-breed at 1 = 74.90%** indica que, para casi 75 de cada 100 imágenes, el vecino más parecido pertenece a la misma raza. Este es un resultado fuerte porque el sistema está comparando embeddings entre **120 razas**.

El resultado de **Same-breed at 5 = 89.94%** indica que, en casi 90 de cada 100 casos, al menos una de las cinco imágenes más parecidas pertenece a la misma raza. Al ampliar a Top-10, el resultado sube a **93.15%**.

Esto muestra que los embeddings capturan información visual relevante para agrupar imágenes de razas similares.

## 3. Interpretación del promedio de vecinos de la misma raza

Además de revisar si aparece al menos un vecino de la misma raza, también se calculó el porcentaje promedio de vecinos del mismo tipo dentro del Top-K.

| Indicador | Resultado |
|---|---:|
| Promedio de vecinos de la misma raza en Top-5 | 72.76% |
| Promedio de vecinos de la misma raza en Top-10 | 71.54% |

Esto significa que, dentro de las cinco imágenes más similares, aproximadamente **72.76%** pertenecen a la misma raza que la imagen consultada. En Top-10, el promedio se mantiene en **71.54%**.

Este resultado es relevante porque no solo demuestra que aparece una imagen correcta dentro del Top-K, sino que la mayoría de los vecinos más cercanos tienden a pertenecer a la misma raza.

En otras palabras, los embeddings no están agrupando imágenes de forma aleatoria; están generando vecindarios visuales coherentes.

## 4. Comparación intra-raza vs inter-raza

| Grupo | Pares | Similitud coseno promedio | Mediana | Desviación estándar | Mínima | Máxima |
|---|---:|---:|---:|---:|---:|---:|
| Intra-raza | 595,823 | 0.7261 | 0.7535 | 0.1413 | 0.1118 | 1.0000 |
| Inter-raza | 20,000 | 0.3160 | 0.2941 | 0.1113 | 0.0684 | 0.9334 |

La comparación más importante del Paso 04 es la diferencia entre similitud intra-raza e inter-raza.

La similitud promedio entre imágenes de la **misma raza** fue de **0.7261**, mientras que la similitud promedio entre imágenes de **razas distintas** fue de **0.3160**.

El margen de separación fue:

| Indicador | Resultado |
|---|---:|
| Intra-raza - Inter-raza | 0.4101 |

Este margen positivo indica que los embeddings separan bien las imágenes de la misma raza frente a imágenes de razas diferentes.

En términos prácticos, las imágenes de perros de la misma raza tienden a estar mucho más cerca entre sí en el espacio de embeddings que las imágenes de razas distintas.

## 5. Métricas complementarias: euclidiana y Manhattan

Aunque la métrica principal del paso es la **similitud coseno**, también se calcularon distancia euclidiana y distancia Manhattan para validar la consistencia de los vecinos encontrados.

| Same breed | Cantidad | Coseno promedio | Distancia euclidiana promedio | Distancia Manhattan promedio |
|---|---:|---:|---:|---:|
| False | 53,525 | 0.8412 | 0.5483 | 5.2395 |
| True | 134,545 | 0.8924 | 0.4520 | 4.2448 |

Los vecinos que pertenecen a la misma raza tienen una **similitud coseno promedio mayor**:  
**0.8924** frente a **0.8412** en vecinos de raza distinta.

Además, presentan distancias menores:

- Euclidiana: **0.4520** para misma raza vs **0.5483** para razas distintas.
- Manhattan: **4.2448** para misma raza vs **5.2395** para razas distintas.

Esto confirma que las métricas complementarias son consistentes con la similitud coseno: los vecinos de la misma raza no solo tienen mayor similitud, sino también menor distancia.

## 6. Análisis de outliers

| Indicador | Resultado |
|---|---:|
| Candidatos a outlier | 1,002 |

Se identificaron **1,002 candidatos a outlier**. Estos casos corresponden a imágenes que tienen baja similitud promedio con otras imágenes de su misma raza.

Un outlier no necesariamente representa un error. Puede indicar:

- Imagen con baja calidad visual.
- Perro parcialmente visible.
- Postura atípica.
- Fondo dominante.
- Iluminación difícil.
- Crop poco representativo.
- Posible etiqueta incorrecta.
- Variabilidad natural dentro de una raza.

Estos outliers son útiles porque permiten identificar casos que podrían revisarse manualmente o usarse para análisis de errores.

## 7. Interpretación general de las métricas

Los resultados del Paso 04 muestran que los embeddings generados en el Paso 03 tienen utilidad real para comparación visual.

La evidencia principal es que la similitud intra-raza promedio fue **0.7261**, mientras que la similitud inter-raza promedio fue **0.3160**. Esta diferencia de **0.4101** demuestra que las imágenes de la misma raza tienden a estar más cerca entre sí que las imágenes de razas diferentes.

Además, el **Same-breed at 5 de 89.94%** indica que, para casi 90% de las imágenes, al menos una de las cinco imágenes más similares pertenece a la misma raza. Esto valida que los embeddings funcionan como base para búsqueda visual.

Las métricas complementarias refuerzan esta interpretación, ya que los vecinos de la misma raza presentan mayor similitud coseno y menores distancias euclidiana y Manhattan.

Por lo tanto, el sistema no solo clasifica razas, sino que también puede encontrar imágenes visualmente parecidas.

## Conclusión final del Paso 04

El Paso 04 validó que los embeddings generados en el Paso 03 son útiles para comparar imágenes de perros mediante similitud visual.

Usando similitud coseno como métrica principal, se evaluaron **18,807 embeddings** de **256 dimensiones** correspondientes a **120 razas**. Los resultados muestran que las imágenes de la misma raza tienen una similitud promedio considerablemente mayor que las imágenes de razas diferentes.

La similitud promedio intra-raza fue de **0.7261**, mientras que la similitud promedio inter-raza fue de **0.3160**, generando un margen de separación de **0.4101**. Esto indica que los embeddings capturan patrones visuales relevantes para distinguir razas y agrupar imágenes similares.

Además, el sistema alcanzó un **Same-breed at 5 de 89.94%** y un **Same-breed at 10 de 93.15%**, lo que demuestra que la mayoría de las imágenes encuentran vecinos visualmente relacionados dentro de sus primeras coincidencias.

Las métricas complementarias de distancia euclidiana y Manhattan confirmaron la consistencia de los resultados, ya que los vecinos de la misma raza tuvieron distancias menores que los vecinos de razas distintas.

Finalmente, se identificaron **1,002 candidatos a outlier**, los cuales pueden ser utilizados para revisión manual o análisis de casos difíciles.

En conclusión, el Paso 04 convierte el modelo en algo más que un clasificador de razas: lo transforma en una herramienta de búsqueda y comparación visual. Esto permite encontrar perros similares, analizar consistencia entre imágenes de la misma raza y preparar el sistema para aplicaciones prácticas como identificación visual, recuperación de imágenes o apoyo en escenarios de perros perdidos.